In [1]:
import os, sys, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

WORK = "/kaggle/working"
DATA_DIR = f"{WORK}/data"
OUT_DIR = f"{WORK}/outputs"
RESULTS_DIR = f"{WORK}/results"
for d in (DATA_DIR, OUT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

def find(name):
    hits = sorted(glob.glob(f"{WORK}/**/{name}", recursive=True)) + \
           sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found: attach the dataset/notebook that makes it (Add Input)")
    if len(hits) > 1:
        print(f"note: {len(hits)} copies of {name}, using {hits[0]}")
    return hits[0]

RAW_CSV = find("afri_med_qa_15k_v2.4_phase_2_15275.csv")

HF_USER = "Tresor26"
def hub_id(run_name):
    return f"{HF_USER}/clinicmate-{run_name}"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO = "/kaggle/temp/ClinicMate"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Tresorshingiro/ClinicMate.git {REPO}
else:
    !git -C {REPO} pull -q
sys.path.append(REPO)

import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)

In [2]:
!pip install -q peft bitsandbytes evaluate rouge_score bert_score
import torch; torch.manual_seed(42)
print(torch.cuda.device_count(), torch.cuda.get_device_name(0))   # should print "1 Tesla T4"

import importlib.metadata as md                                   # copy this output into requirements.txt
for p in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets",
          "evaluate", "sentence-transformers", "scikit-learn", "pandas"]:
    print(f"{p}=={md.version(p)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
1 Tesla T4
torch==2.11.0+cu128
transformers==5.16.1
peft==0.20.0
accelerate==1.14.0
bitsandbytes==0.50.2
datasets==4.8.5
evaluate==0.4.6
sentence-transformers==5.7.0
scikit-learn==1.6.1
pandas==2.3.3


In [3]:
import json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from src.prompts import build_messages
from src.evaluation import mcq_predict, generate_answers, text_metrics, bootstrap_ci

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

def load(base, adapter=None):
    tok = AutoTokenizer.from_pretrained(base)
    model = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb, device_map="auto")
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()
    model.config.use_cache = True
    return model, tok

def run_eval(name, model, tok, retriever=None, eval_file="test_eval",limit=None):
    records = pd.read_json(find(f"{eval_file}.jsonl"), lines=True).to_dict("records")
    if limit: 
        records = records[:limit]
    out, preds_rows = {}, []
    # MCQ: accuracy
    mcq = [r for r in records if r["qtype"] == "mcq"]
    retrieved = [retriever(r) for r in mcq] if retriever else None
    p = mcq_predict(model, tok, mcq, retrieved=retrieved)
    correct = [pi == r["answer_letters"][0] for pi, r in zip(p, mcq)]
    out["mcq_acc"] = float(np.mean(correct)); out["mcq_ci"] = bootstrap_ci(correct)
    for tier in ["expert", "crowdsourced"]:
        c = [ci for ci, r in zip(correct, mcq) if r ["tier"] == tier]
        if c: out[f"mcq_acc_{tier}"] = float(np.mean(c))
    preds_rows += [{**r, "pred": pi, "correct": c} for r, pi, c in zip(mcq, p, correct)]
    # SAQ: generation + ROUGE-L / BERTScore
    sub = [r for r in records if r["qtype"] == "saq"]
    msgs = [ build_messages(r, retriever(r) if retriever else None) for r in sub]
    gen = generate_answers(model, tok, msgs, max_new_tokens=256)
    m = text_metrics(gen, [r["rationale"] for r in sub])
    out["saq"] = {"rougeL": m["rougeL"], "bertscore_f1": m["bertscore_f1"],
                  "avg_words": float(np.mean([len(g.split()) for g in gen]))}
    preds_rows += [{**r, "pred": g, "rougeL": rl, "bertscore": bs}
                  for r, g, rl, bs in zip(sub, gen, m["per_example_rougeL"], m["per_example_bertscore"])]
    json.dump(out, open(f"{RESULTS_DIR}/eval_{name}.json", "w"), indent=2)
    pd.DataFrame(preds_rows).to_json(f"{RESULTS_DIR}/preds_{name}.jsonl", orient="records", lines=True)
    print(name, json.dumps(out, indent=2))
    return out

def evaluate_and_free(name, base, adapter=None, **kw):
    m, t = load(base, adapter); run_eval(name, m, t, **kw)
    del m
    torch.cuda.empty_cache()
    

In [4]:
Q05, Q15 = "Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"
evaluate_and_free("B2_zeroshot_0.5b", Q05)
evaluate_and_free("B3_zeroshot_1.5b", Q15)
evaluate_and_free("E1_qwen0.5b_r16", Q05, hub_id("E1_qwen0.5b_r16")) # load adapters straight from the hub
evaluate_and_free("E2_qwen1.5b_r16", Q15, hub_id("E2_qwen1.5b_r16"))
evaluate_and_free("E3_qwen1.5b_r8", Q15, hub_id("E3_qwen1.5b_r8"))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 37.0MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


E3_qwen1.5b_r8 {
  "mcq_acc": 0.5480378890392422,
  "mcq_ci": [
    0.5128552097428958,
    0.5859269282814614
  ],
  "mcq_acc_expert": 0.5434173669467787,
  "mcq_acc_crowdsourced": 0.68,
  "saq": {
    "rougeL": 0.3019090333339916,
    "bertscore_f1": 0.3161773046674092,
    "avg_words": 37.49382716049383
  }
}


In [5]:
evaluate_and_free("B3_zeroshot_1.5b_official", Q15, eval_file="official_test")
evaluate_and_free("E4_qwen1.5b_official", Q15, hub_id("E4_qwen1.5b_official"), eval_file="official_test")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


B3_zeroshot_1.5b_official {
  "mcq_acc": 0.47450543326831984,
  "mcq_ci": [
    0.4580663137364168,
    0.49150181108943997
  ],
  "mcq_acc_expert": 0.47450543326831984,
  "saq": {
    "rougeL": 0.10118363991960005,
    "bertscore_f1": -0.04527860297785189,
    "avg_words": 132.84457478005865
  }
}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 73.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


E4_qwen1.5b_official {
  "mcq_acc": 0.4845360824742268,
  "mcq_ci": [
    0.46892588464753415,
    0.5012538311507384
  ],
  "mcq_acc_expert": 0.4845360824742268,
  "saq": {
    "rougeL": 0.12886854176026474,
    "bertscore_f1": -0.012483993939825954,
    "avg_words": 32.22873900293255
  }
}


In [6]:
from statsmodels.stats.contingency_tables import mcnemar
def mcnemar_test(a_name, b_name):
    a = pd.read_json(f"{RESULTS_DIR}/preds_{a_name}.jsonl", lines=True).query("qtype=='mcq'")
    b = pd.read_json(f"{RESULTS_DIR}/preds_{b_name}.jsonl", lines=True).query("qtype=='mcq'")
    mm = pd.merge(a[["id", "correct"]], b[["id", "correct"]], on="id", suffixes=("_a", "_b"))
    tbl = pd.crosstab(mm.correct_a, mm.correct_b)
    print(f"{a_name} vs {b_name}\n{tbl}\n{mcnemar(tbl.values, exact=False, correction=True)}\n")

mcnemar_test("B3_zeroshot_1.5b", "E2_qwen1.5b_r16")   
mcnemar_test("E1_qwen0.5b_r16", "E2_qwen1.5b_r16")
mcnemar_test("E2_qwen1.5b_r16", "E3_qwen1.5b_r8")

B3_zeroshot_1.5b vs E2_qwen1.5b_r16
correct_b  0.0  1.0
correct_a          
0.0        273   98
1.0         66  302
pvalue      0.01549094048289175
statistic   5.859756097560975

E1_qwen0.5b_r16 vs E2_qwen1.5b_r16
correct_b  0.0  1.0
correct_a          
0.0        265  178
1.0         74  222
pvalue      8.675839479121049e-11
statistic   42.09920634920635

E2_qwen1.5b_r16 vs E3_qwen1.5b_r8
correct_b  0.0  1.0
correct_a          
0.0        309   30
1.0         25  375
pvalue      0.5896385516265671
statistic   0.2909090909090909



In [7]:
from transformers import pipeline
gate = pipeline('text-classification', model=hub_id("domain-gate-v2"), device=0)
challenge = pd.read_csv(f"{REPO}/gate_challenge.csv")

#OOD-1: full system, i.e refusal rate on OOD and false-refusal rate on medical questions
challenge["gate_medical"] = [r["label"] == "medical" for r in gate(challenge["text"].tolist())]
print("OOD correctly refused:", (~challenge.loc[challenge.label == 0, "gate_medical"]).mean())
print("Medical falsely refused:", (~challenge.loc[challenge.label == 1, "gate_medical"]).mean())
challenge.to_csv(f"{RESULTS_DIR}/gate_v2_challenge_preds.csv", index=False)

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

OOD correctly refused: 0.5666666666666667
Medical falsely refused: 0.0


In [8]:
m, t = load(Q15, hub_id("E2_qwen1.5b_r16"))
ood_q = challenge[challenge.label == 0]["text"].head(15).tolist()
gen = generate_answers(m, t, [build_messages({"qtype": "saq", "question": q, "context": ""}) for q in ood_q])
for q, g in zip(ood_q, gen): print(f"Q: {q}\nA: {g[:300]}\n")
pd.DataFrame({"question": ood_q, "E2_answer": gen}).to_csv(f"{RESULTS_DIR}/ood_generator_only.csv", index=False)
del m
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Q: who won the 2014 world cup
A: Germany won the 2014 World Cup

Q: how do I cook jollof rice
A: 1. Wash the rice thoroughly.
2. In a large pot or pan, heat oil over medium-high heat.
3. Add onions and sauté until they become translucent.
4. Add garlic cloves and continue to sauté for 1 minute.
5. Add tomatoes and cook for 5 minutes.
6. Pour in chicken broth, vegetable stock, and salt.
7. Bring

Q: how do I fix my car's battery
A: Check the battery terminals for corrosion or damage. Clean them with a wire brush dipped in baking soda solution. Replace the battery if it is damaged or old. Check the connections to ensure they are tight. Use a multimeter to check the voltage between the battery terminals and the alternator output

Q: what is the capital of Kenya
A: Nairobi

Q: how do I solve a quadratic equation
A: By factoring, completing the square, or using the quadratic formula.

Q: when did Rwanda gain independence
A: Rwanda gained independence from Belgium on July 1, 1962.

Q: how do

In [9]:
cq = pd.read_json(find("cq_questions.jsonl"), lines=True).sample(30, random_state=7)
cq_recs = cq.to_dict("records")
for name, adapter in [("B3_zeroshot_1.5b", None), ("E2_qwen1.5b_r16", hub_id("E2_qwen1.5b_r16"))]:
    m, t = load(Q15, adapter)
    cq[f"pred_{name}"] = generate_answers(m, t, [build_messages(r) for r in cq_recs])
    del m
    torch.cuda.empty_cache()
cq.to_csv(f"{RESULTS_DIR}/cq_qualitative.csv", index=False)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
best = pd.read_json(f"{RESULTS_DIR}/preds_E2_qwen1.5b_r16.jsonl", lines=True)
keep = ["id", "qtype", "specialty", "question", "rationale", "pred"]
saq_part = best[best.qtype == "saq"].sample(30, random_state=7)[keep]
cqq = pd.read_csv(f"{RESULTS_DIR}/cq_qualitative.csv").head(20)
cq_part = cqq.assign(rationale="(no gold answer)", pred=cqq["pred_E2_qwen1.5b_r16"])[keep]
sheet = pd.concat([saq_part, cq_part])
for c in ["correct_1to5", "omission_y_n", "hallucination_y_n", "harmful_y_n", "africa_context_1to5", "notes"]:
    sheet[c] = ""
sheet.to_csv(f"{RESULTS_DIR}/human_eval_sheet.csv", index=False) 

In [11]:
from sentence_transformers import SentenceTransformer
train = pd.read_json(find("train.jsonl"), lines=True)
kb = train[train["rationale"].str.len() > 0].reset_index(drop=True)
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
kb_vecs = emb.encode(kb["question"].tolist(), batch_size=128, normalize_embeddings=True)

def retriever(r, k=2):
    q = emb.encode([r["question"]], normalize_embeddings=True)[0]
    idx = np.argsort(-(kb_vecs @ q))[:k]
    return kb.loc[idx, ["question", "rationale"]].to_dict("records")

evaluate_and_free("E5_qwen1.5b_r16_rag", Q15, hub_id("E2_qwen1.5b_r16"), retriever=retriever)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


E5_qwen1.5b_r16_rag {
  "mcq_acc": 0.5250338294993234,
  "mcq_ci": [
    0.4884979702300406,
    0.5615696887686062
  ],
  "mcq_acc_expert": 0.5196078431372549,
  "mcq_acc_crowdsourced": 0.68,
  "saq": {
    "rougeL": 0.32328890005353267,
    "bertscore_f1": 0.33923550147501896,
    "avg_words": 41.69547325102881
  }
}
